# Q50 replay-priority training — worker 0

Runs two complete 6,000-update Q50 trainings from the same seed and immutable snapshot. This worker compares **failure-prioritized replay** and **episode-level U20-prioritized replay**. Each batch is 50% ordinary replay and 50% prioritized replay; the critic architecture, Bellman targets, and loss are unchanged. The existing uniform Q50 checkpoint is not loaded.

In [ ]:
EXTRAS = 'train'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')

SNAPSHOT_ID = 'PASTE_PCPCDS_ID_FROM_NOTEBOOK_56'
STRATEGIES = ('failure', 'episode_u20')
MICRO_BATCH_SIZE = 64  # reduce to 32 only if this GPU runs out of memory
CACHE_DOWNLOAD_WORKERS = 8
LABEL_DOWNLOAD_WORKERS = 8
CACHE_ROOT = '/content/qplanning_cache'
OUTPUT_ROOT = '/content/drive/MyDrive/pnp_qplanning_priority'

assert SNAPSHOT_ID.startswith('pcpcds-'), 'paste the notebook 56 pcpcds-* ID'
disk = shutil.disk_usage('/content')
print({
    'worker': 0, 'strategies': STRATEGIES, 'updates_per_strategy': 6000,
    'lr_schedule_horizon': 8000,
    'effective_batch': 64, 'microbatch': MICRO_BATCH_SIZE,
    'train_print_every': 100, 'validate_every': 500,
    'checkpoint_every': 1000, 'priority_fraction': 0.5,
    'local_free_GiB': round(disk.free / 2**30, 1),
    'cache_root': CACHE_ROOT, 'output_root': OUTPUT_ROOT,
})

In [ ]:
from pnp.qplanning_critic import run_q50_priority_worker

reports = run_q50_priority_worker(
    snapshot_id=SNAPSHOT_ID,
    strategies=STRATEGIES,
    cache_root=CACHE_ROOT,
    output_root=OUTPUT_ROOT,
    micro_batch_size=MICRO_BATCH_SIZE,
    cache_download_workers=CACHE_DOWNLOAD_WORKERS,
    label_download_workers=LABEL_DOWNLOAD_WORKERS,
    resume=True,
)
reports